In [2]:
import tkinter as tk  # To have a graphical interface that can handle the cards and interactions.
import os
from astropy.io import fits
import numpy as np
from matplotlib.pyplot import figure, show
import matplotlib.pyplot as plt  # Necessary to close the figure.
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg  # To convert figures to a format Tkinter can display.
from PIL import Image, ImageTk  # To convert and display image to/in Tkinter
import random

class GalaxyObject:
    """
    A class to define the galaxies used.

    Attributes:
        file_path (str): The file path to fetch the FITS file belonging to the input filename.

    Methods:
        plot_galaxy(self): Plots the galaxy to use as an image.
    """
    def __init__(self, filename):
        """Initializes the galaxy object by locating the FITS file"""
        #base_path = os.path.dirname('Images/')  # Get the directory where the images are.
        #full_path = os.path.join(base_path, filename) # Join with filename to make uniform path.
        self.file_path = filename  # Initialize the full path.
        self.hdulist = fits.open(self.file_path)  # Get the HDU list from the file.
    def plot_galaxy(self):
        """A function that plots the galaxy in the FITS file"""
        data = self.hdulist[0].data  # Get image data.
        
        fig = figure(figsize=(8, 8)) # Create a figure.
        frame = fig.add_subplot(1,1,1)
        frame.imshow(data)  # Show data.
        frame.set_axis_off()  # Remove the axes and ticks.
        return fig
    def match_found_fact(self): 
        """A function that gives a random fact when the match between two cards is found"""
        attributes = ['DATE-OBS', 'EXPOSURE', 'TELESCOP', 'OBJCTRA', 'OBJCTDEC'] 
        hdr = self.hdulist[0].header  # Get header.
        fact_object = np.random.choice(attributes)
        if fact_object == 'DATE-OBS':
            fact = hdr['DATE-OBS']
            statement = f"The galaxy was observed on {fact}, at time T."
        elif fact_object == 'EXPOSURE':
            fact = hdr['EXPOSURE']
            statement = f"The galaxy was observed for {fact} minutes."
        elif fact_object == 'TELESCOP':
            fact = hdr['TELESCOP']
            statement = f"The galaxy was observed with the {fact} telescope."
        elif fact_object == 'OBJCTRA':
            RA = hdr['OBJCTRA']
            DEC = hdr['OBJCTDEC']
            statement = f"The galaxy is located at {RA} h min s and {DEC} deg arcmin arcsec"
        elif fact_object == 'OBJCTDEC':
            RA = hdr['OBJCTRA']
            DEC = hdr['OBJCTDEC']
            statement = f"The galaxy is located at {RA} h min s and {DEC} deg arcmin arcsec"
        else:
            statement = "Fact not found."
        return statement

class Card:
    """ 
    A class to create a card for each galaxy.

    Attributes:

    Methods:
    """
    def __init__(self, root, filename):
        self.root = root # Save 'root' window into permanent memory, so other functions can 'çommunicate' with the main window.
        self.galaxy = GalaxyObject(filename)  # Initialize the plot so it can be used inside this class.
        self.is_visible = False  # Variable to see if card is visible.
        self.filename = filename

    def create_image(self):
        figure = self.galaxy.plot_galaxy()  # Plot the galaxy.
        canvas = FigureCanvasTkAgg(figure, master=self.root)  # Translates from matplotlib to Tkinter and puts plot in the window (root).
        canvas.draw()  # Similar to plt.show(): makes sure canvas is actually shown.

        rgba_buffer = canvas.buffer_rgba()  # Get the buffer.
        
        # Translate to RGBA buffer (gives Pillow a view of the plot), then to Tkinter image.
        img = Image.frombuffer('RGBA', canvas.get_width_height(), rgba_buffer, 'raw', 'RGBA', 0, 1)
        img = img.convert('RGB').resize((200,200))  # Convert to RGB to remove transparency and resize to not fill entire screen.
        image = ImageTk.PhotoImage(img)  # Display as image.
        
        plt.close(figure)  # Removes figure from Matplotlib memory to not overload it.
        return image

class Game:
    """
    A class that holds the main game.
    
    Attributes:
    
    Methods:
    """
    def __init__(self, root):
        self.root = root
        self.root.title("Collect the Galaxy")  # Set title (not final title, needs review).
        self.root.geometry('{}x{}'.format(800, 600))  # Define shape of window.
        
        self.files = ["m101.fits", "m101.fits", "bode.fits", "bode.fits", "black_eye.fits", "black_eye.fits", 
                      "fireworks.fits", "fireworks.fits", "cigar.fits", "cigar.fits", "m32_2.fits", "m32_2.fits",
                      "antennae.fits", "antennae.fits", "cent_A.fits", "cent_A.fits"] # Doubled files for a pair
        random.shuffle(self.files)  # Shuffle/reorder the list.
        self.cards = [Card(root, file) for file in self.files]  # Create a card for each file (pairs).
        self.buttons = []  # A list for the buttons so the game remembers them.
        self.flipped_cards = []  # A list for the flipped cards.
        self.is_processing = False  # Prevent clicking while turning back cards
        
        self.title_frame = tk.Frame(self.root, bg='navy', width=400, height=50)
        self.card_frame = tk.Frame(self.root, bg='white', width=400, height=200)
        self.quit_frame = tk.Frame(self.root, bg='gray2', width=400, height=50)

        # Layout: we have rows and columns.
        self.root.grid_rowconfigure(1, weight=1)
        self.root.grid_columnconfigure(0, weight=1)  # Usefull for later for storing the obtained data.

        self.title_frame.grid(row=0, column=0, sticky="ew")
        self.card_frame.grid(row=1, column=0, sticky="nsew")
        self.card_frame.grid_propagate(False)  # Refrains frame from shrinking
        self.quit_frame.grid(row=2, column=0, sticky="ew")

        title_label = tk.Label(self.title_frame, text='COLLECT THE GALAXIES', fg='white', bg='navy', font=('Courier', 20, 'bold'))
        title_label.grid(row=0, column=0, pady=10)  # Add title in window.
        self.title_frame.grid_columnconfigure(0, weight=1) # Center the title

        # Create the buttons by calling the functions that contain a tk.Button command.
        self.create_widgets()
        self.quit_game() 
        
    def create_widgets(self):
        # Create frame (4x4) inside card_frame.
        for x in range(4):
            self.card_frame.grid_columnconfigure(x, weight=1, uniform="group1")  # Make column for x. Uniform states that all buttons must stay the same size.
            self.card_frame.grid_rowconfigure(x, weight=1, uniform="group1")  # Make row for x.
        
        for i, card in enumerate(self.cards):
            button = tk.Button(self.card_frame, text="?", width=20, height=20, command=lambda c=card, i=i: self.on_card_click(c, i))  # Parent window, then parameters. Create function to represent current card.
            button.grid(row=i//4, column=i%4, sticky="nsew", padx=1, pady=1)  # Create grid based on index number
            self.buttons.append(button)   # Append button to remember in game (so it doesn't disappear).
    
    def on_card_click(self, card, index):   
        if not card.is_visible:
            # If the button was clicked on, create an image on it. 
            card_image = card.create_image()  # Generate image for clicked card
            self.buttons[index].configure(image=card_image)  # Update button to image.
            self.buttons[index].image = card_image  
            card.is_visible = True
    
            self.flipped_cards.append((card,index))  # Add the card and its index to the list.
            
        if len(self.flipped_cards) == 2:
            self.is_processing = True 
            self.check_match()

    def check_match(self):
        card_1, index_1 = self.flipped_cards[0]
        card_2, index_2 = self.flipped_cards[1]

        if card_1.filename == card_2.filename:  
            self.root.after(1000, lambda: self.match_found(card_1, card_2, index_1, index_2))  # After 3 seconds, call the function match_found.
        else:
            self.root.after(1000, lambda: self.turn_back(index_1, index_2))  # After one second, call the function turn_back.
    
    def match_found(self, card_1, card_2, index_1, index_2):
        print("You found a match!")
        print(GalaxyObject(card_1.filename).match_found_fact())
        self.buttons[index_1].configure(text="", state="disabled") # Update buttons to no text (to show it was already picked)
        self.buttons[index_2].configure(text="", state="disabled")
        self.cards[index_1].is_visible = False
        self.cards[index_2].is_visible = False
        
        self.flipped_cards = [] # Empty the list
        self.is_processing = False  # Back to starting state
        
    def turn_back(self, index_1, index_2):
        self.buttons[index_1].configure(image="", text="?") # Return buttons to starting state (since the buttons were appended, the program remembers the images and positions).
        self.buttons[index_2].configure(image="", text="?") 
        self.buttons[index_1].image = None
        self.buttons[index_2].image = None
        self.cards[index_1].is_visible = False
        self.cards[index_2].is_visible = False
        self.flipped_cards = []  # Empty the list
        self.is_processing = False  # Back to starting state
        

    def quit_game(self):
        self.quit_frame.grid_columnconfigure(0, weight=1)  # Center the quit button in frame.
        quit_button = tk.Button(self.quit_frame, text="Abort Mission", fg="red", command=self.root.destroy)
        quit_button.grid(row=0, column=0, pady=10)
        self.buttons.append(quit_button)

# Main execution
if __name__ == "__main__":
    root = tk.Tk()  # Create main application window.
    app = Game(root)  # Instance fro the game.
    root.mainloop()  # Keeps window open and waits for user interaction.

TclError: no display name and no $DISPLAY environment variable